# DeepConvNet 改善版（MNIST）

元Notebookの精度低下要因を修正した、Google Colab／CPU／CUDA共通のPyTorch版です。  
学習データでモデルを更新し、検証データでモデル選択し、公式テストデータは最後に一度だけ評価します。

## 原因と元ソースの該当箇所

|No.|原因|元Notebookの箇所|改善|
|---:|---|---|---|
|1|JAX／CuPy／NumPyを同じ`np`として扱い、配列仕様が一致しない|セル7 `import jax.numpy as np`、`import cupy as np`、`import numpy as np`|PyTorchへ統一し、CPU／CUDAだけを明示的に選択|
|2|JAXでは破壊的代入ができない|セル17 `out[self.mask] = 0`、`dout[self.mask] = 0`、セル19・21の配列代入|PyTorch標準層と自動微分へ置換|
|3|手書き`im2col`／`col2im`が遅く、大きなモデルや十分な反復を試しにくい|セル19、21|`nn.Conv2d`、`nn.MaxPool2d`を使用しCUDAにも対応|
|4|モデル容量が小さい|セル23 `16→16→32`、`hidden_size=50`|`32→64→128`、全結合256へ拡張|
|5|BatchNormとDropoutを定義しただけで使用していない|セル17でクラス定義、セル23 `self.layers`には不在|各畳み込みブロックへBatchNormとDropoutを組み込み|
|6|データ拡張がない|セル11は`/255.0`とreshapeのみ|訓練画像だけRandomAffineを適用|
|7|学習量と最適化設定が弱い|セル28 `epochs=5`、`SGD`、`lr=0.001`|AdamW、最大30エポック、Weight Decay、Label Smoothing|
|8|学習率制御、早期終了、最良モデル復元がない|セル25・28|ReduceLROnPlateau、Early Stopping、best checkpoint復元|
|9|ミニバッチを復元抽出して同じ画像が重複し、一部が未使用になる|セル25 `np.random.choice(...)`|DataLoaderの`shuffle=True`で各エポックを非復元シャッフル|
|10|`iter_per_epoch`が小数でエポック判定が不安定|セル25 `train_size / mini_batch_size`と剰余判定|DataLoader単位の明確なエポックループ|
|11|評価が先頭1024件に偏る|セル28 `evaluate_sample_num_per_epoch=1024`|検証データ全件を評価|
|12|独立テストセットがない|セル11で先頭60000件をtrain/validation分割|公式trainをtrain/validationへ分割し、公式testを完全分離|
|13|精度計算が端数バッチを捨てる|セル23 `range(int(x.shape[0] / batch_size))`|DataLoaderですべてのサンプルを処理|
|14|乱数シードが統一されていない|元コードは`train_test_split`だけ固定|Python、NumPy、PyTorch、CUDAを固定|
|15|CSV推論が端数を捨て、CPUでは`.get()`が失敗する|セル34の整数除算、セル35 `y_preds.get()`|全バッチを処理し、`.cpu().numpy()`で共通化|
|16|入力サイズと全結合入力が`32*7*7`に固定|セル23 `W4`|AdaptiveAvgPoolを使用し形状依存を緩和|


In [ ]:
# Colabで不足する場合だけ実行してください。
# !pip install -q torch torchvision scikit-learn pandas matplotlib

## 1. ライブラリ

In [ ]:
from __future__ import annotations

import copy
import os
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms

print("PyTorch:", torch.__version__)

## 2. 設定と再現性

In [ ]:
@dataclass(frozen=True)
class Config:
    seed: int = 42
    data_dir: str = "MNIST_data"
    batch_size: int = 128
    max_epochs: int = 30
    patience: int = 6
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    num_workers: int = 2
    checkpoint: str = "best_deepconvnet_mnist.pt"


CFG = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 3. データ分割と拡張

In [ ]:
MNIST_MEAN = (0.1307,)
MNIST_STD = (0.3081,)

train_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=10,
        translate=(0.10, 0.10),
        scale=(0.90, 1.10),
    ),
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

# 同じインデックスを、訓練時は拡張あり、検証時は拡張なしで読みます。
full_train_aug = datasets.MNIST(CFG.data_dir, train=True, download=True,
                                transform=train_transform)
full_train_eval = datasets.MNIST(CFG.data_dir, train=True, download=True,
                                 transform=eval_transform)
test_set = datasets.MNIST(CFG.data_dir, train=False, download=True,
                          transform=eval_transform)

split_generator = torch.Generator().manual_seed(CFG.seed)
indices = torch.randperm(len(full_train_aug), generator=split_generator).tolist()
val_indices = indices[:10_000]
train_indices = indices[10_000:]
train_set = Subset(full_train_aug, train_indices)
val_set = Subset(full_train_eval, val_indices)

loader_options = {
    "batch_size": CFG.batch_size,
    "num_workers": CFG.num_workers,
    "pin_memory": device.type == "cuda",
    "persistent_workers": CFG.num_workers > 0,
}
train_loader = DataLoader(train_set, shuffle=True, generator=split_generator,
                          **loader_options)
val_loader = DataLoader(val_set, shuffle=False, **loader_options)
test_loader = DataLoader(test_set, shuffle=False, **loader_options)

print("train/validation/test:", len(train_set), len(val_set), len(test_set))

## 4. 改善モデル

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 dropout: float) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ImprovedDeepConvNet(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32, 0.05),   # 28x28 -> 14x14
            ConvBlock(32, 64, 0.10),  # 14x14 -> 7x7
            ConvBlock(64, 128, 0.15), # 7x7 -> 3x3
            nn.AdaptiveAvgPool2d((3, 3)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes),
        )
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # CrossEntropyLossへ渡すためsoftmax前のlogitsを返します。
        return self.classifier(self.features(x))


model = ImprovedDeepConvNet().to(device)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

## 5. 学習・検証

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-5
)
amp_enabled = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()  # BatchNormとDropoutを評価モードへ切り替えます。
    loss_sum = 0.0
    correct = 0
    total = 0
    predictions = []
    targets = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, labels)
        loss_sum += loss.item() * labels.size(0)
        predicted = logits.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        predictions.append(predicted.cpu())
        targets.append(labels.cpu())

    return {
        "loss": loss_sum / total,
        "accuracy": correct / total,
        "predictions": torch.cat(predictions).numpy(),
        "targets": torch.cat(targets).numpy(),
    }


def train_one_epoch(model: nn.Module):
    model.train()
    loss_sum = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return loss_sum / total, correct / total


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

for epoch in range(1, CFG.max_epochs + 1):
    train_loss, train_acc = train_one_epoch(model)
    val_result = evaluate(model, val_loader)
    scheduler.step(val_result["loss"])

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_result["loss"])
    history["val_acc"].append(val_result["accuracy"])

    lr = optimizer.param_groups[0]["lr"]
    print(f"epoch={epoch:02d} lr={lr:.2e} "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4%} "
          f"val_loss={val_result['loss']:.4f} "
          f"val_acc={val_result['accuracy']:.4%}")

    if val_result["loss"] < best_val_loss - 1e-4:
        best_val_loss = val_result["loss"]
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, CFG.checkpoint)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CFG.patience:
            print("Early stopping")
            break

# 最終エポックではなく、検証損失が最小だったパラメータへ戻します。
model.load_state_dict(best_state)

## 6. 独立テスト評価

In [ ]:
# 公式テストセットはモデル選択に使わず、最後に一度だけ評価します。
test_result = evaluate(model, test_loader)
print(f"test_loss={test_result['loss']:.4f}")
print(f"test_accuracy={test_result['accuracy']:.4%}")
print(classification_report(test_result["targets"], test_result["predictions"], digits=4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].legend()
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="validation")
axes[1].set_title("Accuracy")
axes[1].legend()
ConfusionMatrixDisplay.from_predictions(
    test_result["targets"], test_result["predictions"], ax=axes[2], cmap="Blues"
)
axes[2].set_title("Test confusion matrix")
plt.tight_layout()
plt.show()

## 7. ラベルなしCSV推論

In [ ]:
class UnlabeledMNISTCSV(Dataset):
    def __init__(self, csv_path: str | Path) -> None:
        frame = pd.read_csv(csv_path)
        values = frame.to_numpy()
        if values.shape[1] == 785:  # ID列がある場合
            values = values[:, 1:]
        if values.shape[1] != 784:
            raise ValueError(f"784画素列、またはID+784列が必要です: {values.shape}")
        images = torch.from_numpy(values.astype(np.float32)).reshape(-1, 1, 28, 28)
        self.images = (images / 255.0 - MNIST_MEAN[0]) / MNIST_STD[0]

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> torch.Tensor:
        return self.images[index]


@torch.inference_mode()
def predict_csv(input_csv: str | Path, output_csv: str | Path = "y_pred.csv") -> None:
    loader = DataLoader(UnlabeledMNISTCSV(input_csv), batch_size=256, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=device.type == "cuda")
    model.eval()
    batches = []
    for images in loader:  # 最後の端数バッチも処理します。
        logits = model(images.to(device, non_blocking=True))
        batches.append(logits.argmax(1).cpu())
    predictions = torch.cat(batches).numpy()
    pd.DataFrame({"number": predictions}).to_csv(output_csv, index=False)
    print(f"saved: {output_csv}, rows={len(predictions)}")


# 必要な場合だけ実行してください。
# predict_csv("mnist_x_test.csv", "y_pred.csv")

## 改善案の反復評価

採点は実測精度ではなく、コードレビュー上の完成度です。実測値は上の学習・評価セルを実行して確認してください。

### 第1回評価

初回案はCNN強化、AdamW、データ拡張、Early Stoppingを導入しましたが、入力形状固定、混合精度未対応、CSV端数処理の検証不足が残り、平均8.8点でした。

### 第2回評価（最終版）

第1回の不足に対し、AdaptiveAvgPool、AMP、勾配クリッピング、公式テストの完全分離、全件評価、CSV端数処理、CPU/GPU共通変換を追加しました。

|改善案|点数/10|評価理由|
|---|---:|---|
|PyTorchへの計算基盤統一|9.8|配列互換問題を解消し、CPU/CUDAを共通化|
|標準Conv/Pooling・自動微分|9.7|高速化と勾配実装ミスの回避|
|モデル容量拡大|9.4|MNISTには十分な32→64→128構成|
|BatchNorm・Dropout|9.4|学習安定化と過学習抑制を両立|
|データ拡張|9.2|MNISTに適した軽い幾何変換|
|AdamW・Weight Decay・Label Smoothing|9.5|収束性と汎化性能を改善|
|Scheduler・Early Stopping・最良モデル復元|9.7|過学習と不適切な最終モデル採用を防止|
|DataLoader・正しいエポック管理|9.7|復元抽出と小数剰余判定を解消|
|train/validation/test分離・全件評価|9.9|評価リークと偏った1024件評価を防止|
|再現性・AMP・勾配クリッピング|9.3|再現性、速度、数値安定性を改善|
|CSV推論の端数・CPU/GPU対応|9.8|欠落と`.get()`依存を解消|
|AdaptiveAvgPoolによる形状耐性|9.3|全結合入力の固定依存を緩和|
|**平均**|**9.56**|目標の9.0以上を達成|

実測精度を採点に含めるにはNotebookを実行し、`test_accuracy`を確認する必要があります。コードレビューだけで精度値を保証することはできません。
